# Physical-AI-SCA — L3 CW Lab 파일럿

이 노트북이 제3자의 첫 실행 진입점입니다. 기본 설정은 `demo/study.yaml`의 **ISO L3 + CW Lab pilot**이며, 에뮬레이션/실물 × 비마스킹/마스킹 네 실험을 순서대로 수행합니다. CW Lab은 별도 보안 레벨이 아니라 본시험 전 저표본 파일럿입니다. 절차는 ISO급으로 기록하지만 미검출은 표본 부족 상태에서 pass가 아닙니다.

수치와 판정은 결정적 도구가 만들고 Grok은 읽기 전용 자문·출판 감사만 수행합니다. SPA 그림은 이 노트북과 최종 보고서에 표시되지만 사람의 육안 검토는 항상 `pending`입니다.

In [ ]:
from pathlib import Path
import json, subprocess, sys
from IPython.display import display, HTML, SVG

PROJECT = Path.cwd().resolve()
if PROJECT.name == 'demo': PROJECT = PROJECT.parent
assert (PROJECT / 'physai').is_dir(), f'프로젝트 루트를 찾지 못했습니다: {PROJECT}'
sys.path.insert(0, str(PROJECT))
from physai import demo, spec, verify
STUDY = PROJECT / 'demo' / 'study.yaml'
study, experiments = spec.study_experiments(STUDY)
print(study['title'])
print('프로파일:', study['assessment_profile'], '/', study['campaign_stage'])
for meta, item in experiments:
    print(f"- {item['id']}: {meta['role']} / {item['collector']['kind']} / 논리 {sum(x['n'] for x in item['subsets'])}장")

## 수집 전 계약 검토와 Grok 자문
판정 기준과 수량은 수집 전에 고정됩니다. 이 셀은 요청 파일을 만든 뒤 기다리면서 **호스트의 `chipwhisperer-kor` 저장소 루트에서 실행할 정확한 Python 한 줄**을 출력합니다. 그 한 줄을 실행해야 다음 셀로 진행합니다. Grok은 호스트에서 포그라운드 one-shot으로 한 번 실행되고 즉시 종료합니다. 사전 자문이 실패하면 수집을 시작하지 않습니다.

In [ ]:
for _, item in experiments:
    print('\n'.join(spec.summary_lines(item)[:8]), '\n')
assist_path = demo.assist(STUDY, 'pre-collection')
print('Grok 사전 자문:', assist_path)

## 빌드 → 수집 → 파생 생성·분석 → 개별 MD/HTML 보고 → 검증
수집기는 같은 입력의 10회 실행을 `raw-acquisition` HDF5에 각각 보존합니다. 분석기는 원본 해시와 전처리 계약이 맞을 때만 `derived-analysis` HDF5의 float64 평균을 생성·재사용합니다. TA는 원본 실행시간을, 파형 시험은 파생 트레이스를 사용합니다. 실패하면 읽기 전용 Grok 진단을 남긴 뒤 원래 오류를 다시 발생시킵니다.

In [ ]:
def run(command):
    print('\n$', ' '.join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=PROJECT, check=True)

for iut in sorted({x['iut']['name'] for _, x in experiments if x['collector']['kind'] == 'emulation'}):
    run(['make', '-C', 'emul_harness', f'IUT={iut}'])

try:
    for _, item in experiments:
        run([sys.executable, '-m', 'physai.collect', '--study', STUDY, '--experiment', item['id']])
        run([sys.executable, '-m', 'physai.analyze', '--study', STUDY, '--experiment', item['id']])
        run([sys.executable, '-m', 'physai.report', '--run', item['id'], '--study', STUDY])
        run([sys.executable, '-m', 'physai.verify', '--run', item['id'], '--study', STUDY])
except Exception as error:
    try:
        demo.assist(STUDY, 'failure', error)
    except Exception as diagnostic_error:
        print('Grok 실패 자문도 실패했습니다:', diagnostic_error)
    raise

## 결과와 SPA 사람 검토 자료
아래 표와 파형은 최종 검토 입력입니다. 노트북은 육안 검사를 수행했다고 주장하지 않습니다.

In [ ]:
summary_path = demo.write_summary(STUDY)
summary = json.loads(summary_path.read_text(encoding='utf-8'))
try:
    import pandas as pd
    display(pd.DataFrame(summary['comparison_rows']))
except ImportError:
    print(json.dumps(summary['comparison_rows'], ensure_ascii=False, indent=2))
for _, item in experiments:
    figure = PROJECT / 'runs' / item['id'] / 'spa_traces.svg'
    if figure.is_file():
        display(HTML(f'<h3>{item["id"]} — SPA human review: pending</h3>'))
        display(SVG(filename=str(figure)))

## Grok xhigh 출판 감사와 통합 standalone HTML
이 셀도 출판 감사 요청을 만든 뒤 기다리면서 **호스트에서 실행할 정확한 Python 한 줄**을 출력합니다. 같은 한 줄을 다시 실행하십시오. 감사 입력의 상대경로·크기·SHA-256이 하나라도 바뀌면 stale로 거부됩니다. 감사 실패 시 분석 결과는 남지만 publication 상태는 완료되지 않습니다.

In [ ]:
audit_path = demo.grok_audit(STUDY)
published = demo.write_markdown_report(STUDY, audit_path)
print('출판 완료:')
for name, path in published.items(): print(f'- {name}: {path}')
display(HTML(filename=str(published['html'])))

## 연구환경 확장 안내

- L4 또는 full 캠페인으로 바꿀 때는 `demo/study.yaml`의 프로파일/단계를 바꾸고 모든 experiment ID도 새로 만드십시오. 기존 기준과 Dataset을 덮어쓰지 않습니다.
- 새 암호 알고리즘은 `physai.algorithms` 계약의 폭·골든 연산·DPA 분할과 선택적 CPA 모델을 구현하십시오.
- 오실로스코프 등 다른 수집 도구는 공용 HDF5 스키마를 만족하는 Dataset과 실제 장비 메타데이터를 생성해야 합니다. 자세한 실패 조건은 README의 확장 절을 따르십시오.